# Exercise 7: Choosing a donor

*The Computational Phenotype · version 0.1.0*

> You need a Kyrgyz dependency parser and have no Kyrgyz training data. Propose Turkish and Kazakh donor systems, recording their training data, input scripts and annotation conventions, and explain why either might be preferable. Design a small target-language evaluation comparing the available systems and, where supported, native-script and transliterated input. Which result would justify the cost of script conversion?

The notebook runs that evaluation on the gold Kyrgyz test set of the UD-Turkic parallel treebanks (TueCL).

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" pandas stanza requests`. Download the language resources once with `turkicnlp.download(lang, processors=[...])`, as shown in the first code cell. Cells that fetch data from the web need internet access.

In [ ]:
import requests
import stanza
import pandas as pd
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

for lang in ["tur", "kaz"]:
    turkicnlp.download(lang, processors=["tokenize", "pos", "lemma", "depparse"])   # Stanza IMST / KTB models

# Gold Kyrgyz test data (UD Kyrgyz-TueCL); keep word, head and basic relation
URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_Kyrgyz-TueCL/master/ky_tuecl-ud-test.conllu"
sents, cur = [], []
for line in requests.get(URL, timeout=30).text.splitlines():
    if not line.strip():
        if cur: sents.append(cur); cur = []
    elif not line.startswith("#"):
        f = line.split("\t")
        if f[0].isdigit():                                   # skip multiword ranges and empty nodes
            cur.append((f[1], int(f[6]), f[7].split(":")[0]))
if cur: sents.append(cur)
print(len(sents), "sentences,", sum(len(s) for s in sents), "words")

In [ ]:
stanza_dir = str(ModelRegistry.default_dir() / "stanza")
to_cts = Transliterator("kir", Script.CYRILLIC, Script.COMMON_TURKIC)

def evaluate(stanza_lang, convert):
    """UAS/LAS with gold tokenisation; punctuation excluded."""
    nlp = stanza.Pipeline(stanza_lang, processors="tokenize,pos,lemma,depparse", tokenize_pretokenized=True,
                          model_dir=stanza_dir, download_method=None, logging_level="ERROR")
    doc = nlp([[to_cts.transliterate(w) if convert else w for w, _, _ in s] for s in sents])
    uas = las = n = 0
    for gold, pred in zip(sents, doc.sentences):
        for (w, head, rel), pw in zip(gold, pred.words):
            if rel == "punct":
                continue
            n += 1
            uas += pw.head == head
            las += pw.head == head and pw.deprel.split(":")[0] == rel
    return {"UAS": round(100 * uas / n, 1), "LAS": round(100 * las / n, 1), "words scored": n}

rows = [{"system": "Kazakh model, Kyrgyz in Cyrillic", **evaluate("kk", False)},
        {"system": "Turkish model, Kyrgyz converted to CTS", **evaluate("tr", True)},
        {"system": "Turkish model, Kyrgyz in Cyrillic", **evaluate("tr", False)}]
pd.DataFrame(rows)

## Worked solution (run on 18 October 2026)

**Donors.**

| Donor | Training data | Input script | Annotation conventions |
|---|---|---|---|
| Turkish Stanza model | UD Turkish-IMST | Latin | Turkish IMST conventions (e.g. light verbs, copula) |
| Kazakh Stanza model | UD Kazakh-KTB | Cyrillic | Kazakh KTB conventions |

*Why either might be preferable:* Kazakh is in the same Kipchak branch as Kyrgyz, written in the same script, and the two have converged recently; Turkish has much more training data and a larger treebank, but it is Oghuz and written in Latin, so it needs script conversion and its conventions differ more.

**Evaluation.** Gold Kyrgyz TueCL test set: 173 sentences, 1,250 words, of which 1,022 non-punctuation words are scored. Gold tokenisation; UAS/LAS on basic relations.

| System | UAS | LAS |
|---|---|---|
| Kazakh model, Kyrgyz in Cyrillic | 48.8 | 29.2 |
| Turkish model, Kyrgyz converted to CTS | 37.7 | 18.4 |
| Turkish model, Kyrgyz in Cyrillic | 10.8 | 2.0 |

**Interpretation.** Script conversion is essential for the Turkish model: without it, it barely parses Cyrillic input. With conversion it improves greatly but stays behind the Kazakh model, which needs no conversion. Here the related same-script donor wins. Conversion would justify its cost only if the converted Turkish system beat the best native-script option (here the Kazakh model) on the same test sentences by more than the uncertainty of a 173-sentence set; a paired bootstrap over sentences would quantify that. These are short grammar examples translated from Turkish, so check the ranking on natural Kyrgyz text before deciding. An in-language Kyrgyz model (`turkicnlp.download("kir", ...)`, Stanza code `ky`) gives an upper reference point: add `evaluate("ky", False)` to the table.